# Grad-CAM and Grad-CAM++ Visualization for HayabusaNet

This notebook performs post-hoc visual explanation for a trained HayabusaNet model using Grad-CAM and Grad-CAM++.

This Grad-CAM notebook then reconstructs the HayabusaNet architecture from `hayabusanet_model.py`.

## Load HayabusaNet Model 

This cell imports the reusable HayabusaNet model builder and loads the best checkpoint generated by the main training notebook.

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import mixed_precision

from hayabusanet_model import build_hayabusanet


# Keep inference behavior consistent with the main training notebook.
mixed_precision.set_global_policy("mixed_float16")


TARGET_SIZE = (224, 224)  # Must match the input size used during training.
NUM_CLASSES = 4           # Must match the number of output classes in the checkpoint.
MODEL_NAME = "HayabusaNet_Multiclass"
BEST_WEIGHTS_PATH = os.path.join("outputs", f"{MODEL_NAME}_best_weights.keras")


model = build_hayabusanet(
    input_shape=(TARGET_SIZE[0], TARGET_SIZE[1], 3),
    num_classes=NUM_CLASSES,
    model_name=MODEL_NAME
)

if not os.path.exists(BEST_WEIGHTS_PATH):
    raise FileNotFoundError(
        "Best weights file was not found.\n"
        f"Expected path: {BEST_WEIGHTS_PATH}\n\n"
        "Please run HayabusaNet.ipynb first to train the model and generate "
        "the best checkpoint, or manually place the trained weights at the "
        "expected path."
    )

model.load_weights(BEST_WEIGHTS_PATH)

print(f"Loaded model: {model.name}")
print(f"Loaded weights: {BEST_WEIGHTS_PATH}")


## Visualization Configuration and Shared Utilities

This cell defines the input folder, output folder, heatmap settings, and helper functions shared by Grad-CAM and Grad-CAM++. 

In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib as mpl
from PIL import Image


INPUT_DIR = "gradcam_input_images"             # Folder containing images to visualize.
SAVE_DIR = os.path.join("outputs", "gradcam")  # Folder for saved Grad-CAM outputs.
os.makedirs(SAVE_DIR, exist_ok=True)

ALPHA = 0.35       # Heatmap overlay transparency.
CMAP_NAME = "jet"  # Colormap used for Grad-CAM and Grad-CAM++ heatmaps.


def infer_target_size_from_model(model):

    input_shape = model.input_shape

    if isinstance(input_shape, list):
        input_shape = input_shape[0]

    if input_shape[1] is None or input_shape[2] is None:
        raise ValueError("Model input height and width must be defined.")

    return int(input_shape[1]), int(input_shape[2])


TARGET_SIZE = infer_target_size_from_model(model)


def find_last_spatial_layer(model):
    """Find the last spatial feature layer before global average pooling.

    Args:
        model: Trained Keras model.

    Returns:
        Name of the selected spatial layer.

    Raises:
        ValueError: If no suitable four-dimensional spatial layer is found.
    """
    passed_gap = False

    
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.GlobalAveragePooling2D):
            passed_gap = True
            continue

        if passed_gap:
            output_shape = getattr(layer, "output_shape", None)
            if output_shape is None and hasattr(layer, "output"):
                output_shape = tuple(layer.output.shape)

            if isinstance(output_shape, tuple) and len(output_shape) == 4:
                return layer.name

    
    for layer in reversed(model.layers):
        output_shape = getattr(layer, "output_shape", None)
        if output_shape is None and hasattr(layer, "output"):
            output_shape = tuple(layer.output.shape)

        if isinstance(output_shape, tuple) and len(output_shape) == 4:
            return layer.name

    raise ValueError("No suitable 4D spatial layer was found.")


last_spatial_name = find_last_spatial_layer(model)
print("[Grad-CAM] Last spatial layer:", last_spatial_name)


def load_image_as_model_input(path, img_size=TARGET_SIZE):
    """Load an image and convert it into model input format.

    The image is decoded as RGB, resized to the model input size, and kept in
    the [0, 255] range because normalization is handled by the model's
    Rescaling layer.

    Args:
        path: Path to the input image.
        img_size: Target image size as a tuple of height and width.

    Returns:
        Tensor with shape [1, H, W, 3].
    """
    x = tf.io.read_file(path)
    x = tf.io.decode_image(x, channels=3, expand_animations=False)
    x = tf.image.resize(x, img_size, method=tf.image.ResizeMethod.BILINEAR)
    x = tf.cast(x, tf.float32)

    return tf.expand_dims(x, 0)


def get_class_index(target_class_index, preds):
    """Resolve the class index used for visual explanation.

    Args:
        target_class_index: Optional class index. If None, the predicted class is used.
        preds: Model prediction tensor.

    Returns:
        Tensor scalar containing the class index.
    """
    if target_class_index is None:
        return tf.cast(tf.argmax(preds[0]), tf.int32)

    return tf.cast(target_class_index, tf.int32)


def normalize_heatmap(heatmap, eps=1e-12):
    """Normalize a heatmap into the [0, 1] range.

    Args:
        heatmap: Heatmap tensor.
        eps: Small value used for numerical stability.

    Returns:
        Normalized heatmap tensor.
    """
    # ReLU keeps only positive evidence, while epsilon prevents division by zero.
    heatmap = tf.nn.relu(heatmap)
    maxv = tf.reduce_max(heatmap)

    return tf.where(maxv > eps, heatmap / (maxv + eps), tf.zeros_like(heatmap))


def colorize_and_overlay(rgb_uint8_HWC, heatmap_01_HW, alpha=ALPHA):

    cmap = cm.get_cmap(CMAP_NAME)
    cm_rgb = (cmap(heatmap_01_HW)[..., :3] * 255).astype(np.uint8)
    overlay = (rgb_uint8_HWC * (1 - alpha) + cm_rgb * alpha).astype(np.uint8)

    return Image.fromarray(overlay)


def load_rgb_background(path, img_size=TARGET_SIZE):

    raw = tf.io.read_file(path)
    raw = tf.io.decode_image(raw, channels=3, expand_animations=False)
    raw = tf.image.resize(raw, img_size, method=tf.image.ResizeMethod.BILINEAR)

    return tf.clip_by_value(tf.cast(raw, tf.uint8), 0, 255).numpy()


def save_colorbar_legend(
    save_dir,
    filename="gradcam_legend_jet.png",
    orientation="vertical",
    height_in=4.0,
    width_in=0.6
):

    # Figure dimensions are adjusted based on the selected legend orientation.
    if orientation == "vertical":
        figsize = (width_in, height_in)
    else:
        figsize = (height_in, 1.0)

    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.axis("off")

    # Create a normalized scalar map so the legend matches the [0, 1] heatmap scale.
    sm = mpl.cm.ScalarMappable(
        norm=mpl.colors.Normalize(0, 1),
        cmap=CMAP_NAME
    )
    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation=orientation,
        fraction=0.8,
        pad=0.02
    )

    cbar.set_label("Grad-CAM importance (0 = low, 1 = high)", fontsize=10)

    if orientation == "vertical":
        cbar.set_ticks([0.0, 0.25, 0.5, 0.75, 1.0])
        cbar.set_ticklabels(["Low", "", "Medium", "", "High"])
    else:
        cbar.set_ticks([0.0, 0.25, 0.5, 0.75, 1.0])
        cbar.ax.set_xticklabels(["Low", "", "Medium", "", "High"])

    out_path = os.path.join(save_dir, filename)
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

    print(f"[Grad-CAM] Legend saved: {out_path}")


## Grad-CAM

This section generates Grad-CAM heatmaps by computing the gradient of the predicted class score with respect to the selected spatial feature map. The resulting heatmap highlights image regions that contribute positively to the model prediction.


In [ ]:
def make_gradcam_heatmap(
    img_tensor,
    model,
    last_layer_name,
    target_class_index=None,
    out_size=TARGET_SIZE
):

    img_tensor = tf.cast(img_tensor, tf.float32)
    last_layer = model.get_layer(last_layer_name)

    # The gradient model returns both the spatial feature map and the final prediction.
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[last_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_tensor, training=False)
        class_index = get_class_index(target_class_index, preds)
        class_score = tf.gather(preds[0], class_index)

    grads = tape.gradient(class_score, conv_out)

    if grads is None:
        raise ValueError("Grad-CAM gradients are None for this image.")

    conv_out = tf.cast(conv_out, tf.float32)[0]
    grads = tf.cast(grads, tf.float32)[0]

    # Channel weights are obtained by global-average pooling the gradients.
    weights = tf.reduce_mean(grads, axis=(0, 1))
    heatmap = tf.reduce_sum(conv_out * weights, axis=-1)
    heatmap = normalize_heatmap(heatmap)
    heatmap = tf.image.resize(heatmap[..., None], out_size)

    return tf.squeeze(heatmap, -1).numpy(), int(class_index.numpy()), preds.numpy()[0]


def save_gradcam_for_image(
    img_path,
    model,
    last_layer_name,
    save_dir,
    alpha=ALPHA,
    target_class_index=None
):

    x = load_image_as_model_input(img_path, TARGET_SIZE)
    raw_uint8 = load_rgb_background(img_path, TARGET_SIZE)

    heatmap, used_cls, probs = make_gradcam_heatmap(
        x,
        model,
        last_layer_name,
        target_class_index,
        TARGET_SIZE
    )

    pred_cls = int(np.argmax(probs))
    pred_prob = float(probs[pred_cls])

    overlay_img = colorize_and_overlay(raw_uint8, heatmap, alpha=alpha)
    base = os.path.splitext(os.path.basename(img_path))[0]
    out_name_overlay = f"{base}_pred{pred_cls}_p{pred_prob:.3f}_t{used_cls}.png"
    overlay_img.save(os.path.join(save_dir, out_name_overlay))

    return pred_cls, pred_prob, used_cls


## Grad-CAM++

This section generates Grad-CAM++ heatmaps using higher-order gradients.

In [ ]:
def make_gradcampp_heatmap(
    img_tensor,
    model,
    last_layer_name,
    target_class_index=None,
    out_size=TARGET_SIZE
):

    img_tensor = tf.cast(img_tensor, tf.float32)
    last_layer = model.get_layer(last_layer_name)

    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[last_layer.output, model.output]
    )

    
    with tf.GradientTape(persistent=True) as tape:
        conv_out, preds = grad_model(img_tensor, training=False)
        class_index = get_class_index(target_class_index, preds)
        class_score = tf.gather(preds[0], class_index)

        grads_1 = tape.gradient(class_score, conv_out)
        grads_2 = tape.gradient(grads_1, conv_out) if grads_1 is not None else None
        grads_3 = tape.gradient(grads_2, conv_out) if grads_2 is not None else None

    del tape

    if grads_1 is None or grads_2 is None or grads_3 is None:
        raise ValueError("Grad-CAM++ higher-order gradients are None for this image.")

    conv_out = tf.cast(conv_out, tf.float32)[0]
    grads_1 = tf.cast(grads_1, tf.float32)[0]
    grads_2 = tf.cast(grads_2, tf.float32)[0]
    grads_3 = tf.cast(grads_3, tf.float32)[0]

    eps = tf.constant(1e-12, dtype=tf.float32)  # Prevents division by zero.

    # Grad-CAM++ alpha coefficients weight positive gradients at each spatial location.
    alpha_num = grads_2
    alpha_denom = 2.0 * grads_2 + conv_out * grads_3
    alpha = tf.where(
        tf.abs(alpha_denom) > eps,
        alpha_num / (alpha_denom + eps),
        tf.zeros_like(alpha_denom)
    )

    relu_grads = tf.nn.relu(grads_1)
    weights_ijk = alpha * relu_grads
    weights_k = tf.reduce_sum(weights_ijk, axis=(0, 1))

    heatmap = tf.reduce_sum(conv_out * weights_k, axis=-1)
    heatmap = normalize_heatmap(heatmap, eps=eps)
    heatmap = tf.image.resize(heatmap[..., None], out_size)

    return tf.squeeze(heatmap, -1).numpy(), int(class_index.numpy()), preds.numpy()[0]


def save_gradcampp_for_image(
    img_path,
    model,
    last_layer_name,
    save_dir,
    alpha=ALPHA,
    target_class_index=None,
    suffix="_GCPP"
):

    x = load_image_as_model_input(img_path, TARGET_SIZE)
    raw_uint8 = load_rgb_background(img_path, TARGET_SIZE)

    heatmap, used_cls, probs = make_gradcampp_heatmap(
        x,
        model,
        last_layer_name,
        target_class_index,
        TARGET_SIZE
    )

    pred_cls = int(np.argmax(probs))
    pred_prob = float(probs[pred_cls])

    overlay_img = colorize_and_overlay(raw_uint8, heatmap, alpha=alpha)
    base = os.path.splitext(os.path.basename(img_path))[0]
    out_name = f"{base}_pred{pred_cls}_p{pred_prob:.3f}_t{used_cls}{suffix}.png"
    overlay_img.save(os.path.join(save_dir, out_name))

    return pred_cls, pred_prob, used_cls


## Batch Visualization

In [ ]:
# Save the colorbar legend once before processing all images.
save_colorbar_legend(
    SAVE_DIR,
    filename="gradcam_legend_jet.png",
    orientation="vertical",
    height_in=4.0,
    width_in=0.6
)

# Collect all supported image files recursively to avoid manual sample selection.
valid_exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
image_paths = sorted([
    p for p in glob.glob(os.path.join(INPUT_DIR, "**", "*"), recursive=True)
    if p.lower().endswith(valid_exts)
])

print(f"[Grad-CAM] Scanning {len(image_paths)} images from: {INPUT_DIR}")

ok, fail = 0, 0
for p in image_paths:
    try:
        _ = save_gradcam_for_image(
            p,
            model,
            last_spatial_name,
            SAVE_DIR,
            alpha=ALPHA,
            target_class_index=None
        )

        _ = save_gradcampp_for_image(
            p,
            model,
            last_spatial_name,
            SAVE_DIR,
            alpha=ALPHA,
            target_class_index=None
        )

        ok += 1
    except Exception as e:
        print(f"[WARN] Fail on {p}: {e}")
        fail += 1

print(f"[Grad-CAM(+/++)] Done. Success={ok}, Fail={fail}. Saved in: {SAVE_DIR}")
